# 01 — De l'ACP à l'auto-encodeur

Mode utilisé dans ce support : **smoke**, destiné à vérifier la chaîne complète. Les valeurs commentées proviennent du rapport calculé, jamais de scores inventés.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "outputs" / "reports" / "results_smoke.json"
if not REPORT.exists():
    raise FileNotFoundError("Exécutez d'abord : .venv\\Scripts\\python scripts\\run_smoke.py --mode smoke")
results = json.loads(REPORT.read_text(encoding="utf-8"))
sns.set_theme(style="whitegrid")

## Question étudiée et intuition

Une représentation comprimée peut-elle reconstruire une journée de 48 points ? L'encodeur calcule $z=f_\theta(x)$, le décodeur $\hat x=g_\phi(z)$, et l'apprentissage minimise ici la MSE sur les seuls points initialement observés :

$$\mathcal L(\theta,\phi)=\frac{\sum_{i,t}m_{it}(x_{it}-\hat x_{it})^2}{\sum_{i,t}m_{it}}.$$

Les mini-lots fournissent des gradients bruités mais économiques. La validation et l'arrêt anticipé limitent le surapprentissage. Un auto-encodeur linéaire, sous des hypothèses adaptées, retrouve le **sous-espace** principal de l'ACP ; ses axes latents ne sont pas nécessairement les composantes principales elles-mêmes.

## Architecture et boucle d'entraînement explicites

Le pipeline principal entraîne plusieurs modèles et conserve leurs métriques. Ici, on refait une petite expérience en cellules : lecture des journées, séparation par **foyer**, normalisation sur le train, mini-lots, encodeur/décodeur, puis MSE train et validation. Ce code est volontairement lisible ; le notebook 07 reprend le raisonnement à l'échelle annuelle.

In [ ]:
import json, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from cer_ae.preprocessing import normalize_common

raw = np.load(ROOT/'data/processed/smoke_daily_windows.npz')['values']
meta = pd.read_parquet(ROOT/'data/processed/smoke_daily_metadata.parquet')
splits = json.loads((ROOT/'data/processed/smoke_splits.json').read_text(encoding='utf-8'))
indices = {part: np.flatnonzero(meta.client_id.isin(ids)) for part, ids in splits.items()}
assert not (set(splits['train']) & set(splits['validation']))
x, scaler = normalize_common(raw, indices['train'])
train_loader = DataLoader(TensorDataset(torch.tensor(x[indices['train']])), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(x[indices['validation']])), batch_size=128)
print('Journées × 48 :', x.shape, ' ; mini-lot :', next(iter(train_loader))[0].shape)

In [ ]:
class SmallAE(nn.Module):
    def __init__(self, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(48, 32), nn.ReLU(), nn.Linear(32, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 48))

    def forward(self, curves):
        return self.decoder(self.encoder(curves))

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallAE(latent_dim=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = {'train': [], 'validation': []}
for epoch in range(10):
    model.train()
    train_sum, train_n = 0., 0
    for (curves,) in train_loader:
        curves = curves.to(device)
        optimizer.zero_grad()
        loss = ((model(curves) - curves)**2).mean()
        loss.backward()
        optimizer.step()
        train_sum += loss.item() * len(curves)
        train_n += len(curves)
    model.eval()
    val_sum, val_n = 0., 0
    with torch.no_grad():
        for (curves,) in val_loader:
            curves = curves.to(device)
            val_sum += ((model(curves) - curves)**2).mean().item() * len(curves)
            val_n += len(curves)
    history['train'].append(train_sum / train_n)
    history['validation'].append(val_sum / val_n)
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(history['train'], label='train')
ax.plot(history['validation'], label='validation')
ax.set(xlabel='Époque', ylabel='MSE standardisée', title='Entraînement explicite du petit AE')
ax.legend(); plt.show()

In [ ]:
rows=[]
for name, d in results['daily'].items():
    rows.append({'méthode':name, 'RMSE observée':d['rmse_observed'], 'MAE observée':d['mae_observed'],
                 'paramètres':d.get('parameters'), 'secondes':d.get('history',{}).get('seconds')})
comparison=pd.DataFrame(rows).sort_values('RMSE observée')
display(comparison)
ax=comparison.plot.bar(x='méthode', y='RMSE observée', figsize=(10,3), legend=False)
ax.set_ylabel('RMSE standardisée (test)'); plt.xticks(rotation=35,ha='right'); plt.show()

In [ ]:
histories={k:v['history'] for k,v in results['daily'].items() if 'history' in v}
fig,ax=plt.subplots(figsize=(8,3))
for name,h in histories.items(): ax.plot(h['val_loss'],marker='o',label=name)
ax.set(xlabel='Époque',ylabel='MSE validation',title='Courbes d’apprentissage (smoke)'); ax.legend(ncol=2); plt.show()
print('Interprétation calculée :', comparison.iloc[0]['méthode'], 'obtient la plus faible RMSE dans ce smoke test.')

In [ ]:
fig,ax=plt.subplots(figsize=(10,3))
for name in ['pca_2','linear_ae_2','nonlinear_ae_2']:
    ax.plot(np.arange(48)/2,results['daily'][name]['rmse_by_slot'],label=name)
ax.set(xlabel='Heure',ylabel='RMSE standardisée',title='Erreur selon l’heure'); ax.legend(); plt.show()
latent=np.load(ROOT/'data/processed/smoke_daily_latent.npy'); meta=pd.read_parquet(ROOT/'data/processed/smoke_daily_metadata.parquet')
fig,ax=plt.subplots(figsize=(6,4)); sns.scatterplot(x=latent[:,0],y=latent[:,1],hue=meta.season,s=12,alpha=.6,ax=ax); ax.set_title('Espace latent, coloré par saison'); plt.show()

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT/'outputs/figures/smoke_reconstructions.png')))

## Lecture prudente

Deux époques vérifient le pipeline, elles ne suffisent pas à conclure sur la capacité maximale des réseaux. Dans l'exécution observée, l'ACP est devant les auto-encodeurs. C'est compatible avec un signal assez régulier et des réseaux encore sous-entraînés.

Questions : la MSE favorise-t-elle les pics ou les creux ? Comment comparer équitablement les temps de calcul CPU/GPU ? Pourquoi faut-il projeter les latents en 2D plutôt que prendre arbitrairement deux coordonnées d'un latent de dimension 16 ?